In [1]:
import os
import torch
import numpy as np
import pandas as pd

from easydict import EasyDict as edict

from data import prepare_splits_ood, prepare_splits_id, domain_cutoff
from eval import compute_all_metrics


dataset_realworld = [
    "absenteeism",
    "airlines",
    "ames_housing_prices",
    "chess",
    "cleveland_heart_disease",
    "diabetes_130_us_hospitals",
    "diabetes_questionaire",
    "electricity",
    "free_light_chain_mortality",
    "indian_liver_patient",
    "istanbul_stock_exchange",
    "parking_birmingham",
    "pima_indians_diabetes",
    "occupancy_detection",
    "urban_traffic_sao_paulo"
]

dataset_synthetic = [
    "binary_label_shift_dataset_2_classes_200_samples_10_domains",
    "drain_rotated_two_moons",
    "intersecting_blobs_dataset_3_classes_120_samples_14_domains",
    "moving_blobs_dataset_2d_2_classes_moving_diagonal_line",
    "moving_blobs_dataset_2d_4_classes_moving_square",
    "randomrbf_drift",
    "rotated_five_blobs_20_deg",
    "rotating_hyperplane",
    "rotating_segments",
    "shifting_two_spirals",
    "sin_classification",
    "sliding_circle"
]

In [2]:
from importlib import resources

import tabpfn
from tabpfn.scripts.tabular_baselines import transformer_metric
from tabpfn.best_models import get_best_tabpfn, TabPFNModelPathsConfig

# Get the library path for the tabpfn package
libpath = resources.files(tabpfn)


# Helper function to load each pre-trained model with the corresponding configuration.
def get_model(task_type, model_path, model_type):
    model_path_config = TabPFNModelPathsConfig(
        paths=[f"{libpath}/model_cache/{model_path}.cpkt"], task_type=task_type
    )

    model = get_best_tabpfn(
        task_type=task_type,
        model_type=model_type,
        paths_config=model_path_config,
        debug=False,
        device="auto"
    )
    model.show_progress = False
    model.seed = 1

    return model


task_type = "dist_shift_multiclass"

models_to_load = [
    ("tabpfn_dist_model_1", "best_dist"),
    # ("tabpfn_dist_model_2", "best_dist"),
    # ("tabpfn_dist_model_3", "best_dist"),
    # ("tabpfn_dist_ablation_no_t2v_model_1", "best_dist"),
    # ("tabpfn_base_model_1", "best_base"),
    # ("tabpfn_base_model_2", "best_base"),
    # ("tabpfn_base_model_3", "best_base"),
]

# Load each model
models = {
    model_name: get_model(task_type, model_name, model_type)
    for model_name, model_type in models_to_load
}

In [3]:
config = edict()
config.domain_col = 'Domain'
config.target = 'Label'
config.train_ratio = 0.8
config.val_ratio = 0.1


mode = 'ood'
clf = models["tabpfn_dist_model_1"]
for d_name in dataset_realworld:
    print(d_name)
    path = f"{os.environ['HOME']}/workspace/research/STRIDE-main/data/real-world/{d_name}/{d_name}_prep.csv"

    df = pd.read_csv(path)
    feature_cols = [c for c in df.columns if c not in ('Label')]

    if mode == "ood":
        cutoff = domain_cutoff(df, config)

    metrics_results = []
    for seed in [42, 123, 7, 2026, 99]:
        if mode == "ood":
            X_train, y_train, X_val, y_val, X_test, y_test = prepare_splits_ood(df, config, feature_cols, cutoff, seed)
        else:
            X_train, y_train, X_val, y_val, X_test, y_test = prepare_splits_id(df, config, feature_cols, seed)

        X_train = pd.concat([X_train, X_val]).reset_index(drop=True)

        dist_shift_domain_train = torch.Tensor(X_train['Domain'].values)
        dist_shift_domain_test = torch.Tensor(X_test['Domain'].values)
        X_train = torch.Tensor(X_train.drop(['Domain'], axis=1).values)
        X_test = torch.Tensor(X_test.drop(['Domain'], axis=1).values)
        y_train = torch.Tensor(pd.concat([y_train, y_val]).values)

        clf.fit(X_train, y_train, additional_x={"dist_shift_domain": dist_shift_domain_train})
        preds = clf.predict_proba(X_test, additional_x={"dist_shift_domain":dist_shift_domain_test})
        metrics = compute_all_metrics(y_test, preds)
        for k, v in metrics.items():
            print(f'{k},', v, end=', ')
        print()
        metrics_results.append(metrics)
    averages = {key: sum(d[key] for d in metrics_results) / len(metrics_results) for key in metrics_results[0]}
    for k, v in averages.items():
        print(f'{k},', v, end=', ')


absenteeism
Initialized decoder for standard with (None, 10)  and nout 10
acc, 0.5595238095238095, 
roc_auc, 0.7815935476672122, 
f1, 0.46641422609732464, 
ece, 0.11451655200549533, 
acc, 0.5595238095238095, 
roc_auc, 0.7816695879713359, 
f1, 0.46641422609732464, 
ece, 0.11552358915408455, 


KeyboardInterrupt: 

In [ ]:
mode = 'ood'
clf = models["tabpfn_dist_model_1"]
for d_name in dataset_synthetic:
    print(d_name)
    path = f"/home/jovyan/workspace/research/STRIDE-main/data/synthetic/{d_name}/{d_name}.csv
    df = pd.read_csv(path)
    feature_cols = [c for c in df.columns if c not in ('Label')]

    if mode == "ood":
        cutoff = domain_cutoff(df, config)

    metrics_results = []
    for seed in [42, 123, 7, 2026, 99]:
        if mode == "ood":
            X_train, y_train, X_val, y_val, X_test, y_test = prepare_splits_ood(df, config, feature_cols, cutoff, seed)
        else:
            X_train, y_train, X_val, y_val, X_test, y_test = prepare_splits_id(df, config, feature_cols, seed)

        X_train = pd.concat([X_train, X_val]).reset_index(drop=True)

        dist_shift_domain_train = torch.Tensor(X_train['Domain'].values)
        dist_shift_domain_test = torch.Tensor(X_test['Domain'].values)
        X_train = torch.Tensor(X_train.drop(['Domain'], axis=1).values)
        X_test = torch.Tensor(X_test.drop(['Domain'], axis=1).values)
        y_train = torch.Tensor(pd.concat([y_train, y_val]).values)

        clf.fit(X_train, y_train, additional_x={"dist_shift_domain": dist_shift_domain_train})
        preds = clf.predict_proba(X_test, additional_x={"dist_shift_domain":dist_shift_domain_test})
        metrics = compute_all_metrics(y_test, preds)
        for k, v in metrics.items():
            print(f'{k},', v, end=', ')
        print()
        metrics_results.append(metrics)
    averages = {key: sum(d[key] for d in metrics_results) / len(metrics_results) for key in metrics_results[0]}
    for k, v in averages.items():
        print(f'{k},', v, end=', ')
